In [7]:
!gcloud container clusters create pytorch-inference-cluster \
  --machine-type=e2-standard-4 \
  --zone=asia-southeast1-a \
  --enable-autoscaling \
  --min-nodes=1 \
  --max-nodes=3 \
  --num-nodes=1 

Note: Your Pod address range (`--cluster-ipv4-cidr`) can accommodate at most 1008 node(s).
Creating cluster pytorch-inference-cluster in asia-southeast1-a... Cluster is b
eing configured...⠹                                                            
Creating cluster pytorch-inference-cluster in asia-southeast1-a...⠧^C          
Creating cluster pytorch-inference-cluster in asia-southeast1-a...aborted by ct
rl-c.
ERROR: (gcloud.container.clusters.create) Aborted by user.


In [55]:
!gcloud container clusters delete pytorch-inference-cluster \
  --zone=asia-southeast1-a \
  --quiet

Deleting cluster pytorch-inference-cluster...⠼^C                               
Deleting cluster pytorch-inference-cluster...aborted by ctrl-c.                
ERROR: (gcloud.container.clusters.delete) Aborted by user.


In [56]:
!gcloud container clusters list --format="table(name, location, masterVersion, status, nodePools.name.list():label=NODE_POOLS)"

In [24]:
!gcloud container clusters get-credentials pytorch-inference-cluster \
  --zone asia-southeast1-a

Fetching cluster endpoint and auth data.
kubeconfig entry generated for pytorch-inference-cluster.


In [25]:
!kubectl config current-context

gke_ridwan-faturrahman_asia-southeast1-a_pytorch-inference-cluster


In [5]:
!gcloud storage ls 'gs://my-model-training/result-mar'

gs://my-model-training/result-mar/model.mar


In [6]:
serving_container_uri = "us-docker.pkg.dev/vertex-ai/prediction/pytorch-cpu.1-11:latest"

In [26]:
!kubectl create namespace torchserve

namespace/torchserve created


In [27]:
!kubectl apply -f torchserve-config.yaml

configmap/torchserve-config created


In [28]:
!kubectl apply -f torchserve-deployment.yaml

deployment.apps/torchserve created


In [30]:
!kubectl apply -f torchserve-service.yaml

service/torchserve created


In [35]:
!kubectl get svc -n torchserve -w

NAME         TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)                       AGE
torchserve   LoadBalancer   34.118.236.10   <pending>     80:30366/TCP,8081:31411/TCP   59s
torchserve   LoadBalancer   34.118.236.10   <pending>     80:30366/TCP,8081:31411/TCP   64s
torchserve   LoadBalancer   34.118.236.10   34.143.238.97   80:30366/TCP,8081:31411/TCP   64s
^C


In [43]:
!kubectl get svc -n torchserve

NAME         TYPE           CLUSTER-IP      EXTERNAL-IP     PORT(S)                       AGE
torchserve   LoadBalancer   34.118.236.10   34.143.238.97   80:30366/TCP,8081:31411/TCP   4m17s


In [40]:
!kubectl set env deployment/torchserve -n torchserve \
  AIP_STORAGE_URI=gs://my-model-training/result-mar \
  AIP_HTTP_PORT=8080 \
  AIP_HEALTH_ROUTE=/ping \
  AIP_PREDICT_ROUTE=/predictions/model

deployment.apps/torchserve env updated


In [42]:
!kubectl get pods -n torchserve

NAME                          READY   STATUS             RESTARTS      AGE
torchserve-68b4c8674f-hlkqd   0/1     CrashLoopBackOff   4 (54s ago)   4m28s
torchserve-6b4698cc44-ftw5h   1/1     Running            0             56s
torchserve-6b4698cc44-knlrv   0/1     Running            0             19s


In [44]:
!curl http://34.143.238.97/ping

{
  "status": "Healthy"
}


In [52]:
import requests

url = "http://34.143.238.97/predictions/model"

# payload = {
#     "instances": [
#         {
#             "body": {
#                 "data": [
#                     [1.0, 2.0, 3.0, 4.0],
#                     [1.0, 2.0, 3.0, 9.0]
#                 ]
#             }
#         }        
#     ]
# }
payload = {
    "data": [
        [1.0, 2.0, 3.0, 4.0],
        [1.0, 2.0, 3.0, 9.0]
    ]
}

response = requests.post(
    url,
    json=payload,
    headers={"Content-Type": "application/json"}
)

In [53]:
response.status_code

200

In [54]:
result = response.json()
result

[[2.3237223625183105], [3.3951992988586426]]

In [48]:
!kubectl logs -n torchserve -l app=torchserve --tail=50

Defaulted container "torchserve" out of: torchserve, download-model (init)
Defaulted container "torchserve" out of: torchserve, download-model (init)
2026-08-27T07:18:37,182 [INFO ] pool-3-thread-1 TS_METRICS - DiskUsage.Gigabytes:13.206829071044922|#Level:Host|#hostname:torchserve-6b4698cc44-ftw5h,timestamp:1787815117
2026-08-27T07:18:37,182 [INFO ] pool-3-thread-1 TS_METRICS - DiskUtilization.Percent:14.0|#Level:Host|#hostname:torchserve-6b4698cc44-ftw5h,timestamp:1787815117
2026-08-27T07:18:37,182 [INFO ] pool-3-thread-1 TS_METRICS - MemoryAvailable.Megabytes:14199.97265625|#Level:Host|#hostname:torchserve-6b4698cc44-ftw5h,timestamp:1787815117
2026-08-27T07:18:37,182 [INFO ] pool-3-thread-1 TS_METRICS - MemoryUsed.Megabytes:1422.3515625|#Level:Host|#hostname:torchserve-6b4698cc44-ftw5h,timestamp:1787815117
2026-08-27T07:18:37,182 [INFO ] pool-3-thread-1 TS_METRICS - MemoryUtilization.Percent:11.2|#Level:Host|#hostname:torchserve-6b4698cc44-ftw5h,timestamp:1787815117
2026-08-27T07:18

In [20]:
# curl -X POST http://EXTERNAL_IP/predictions/model \
#   -H "Content-Type: application/json" \
#   -d '{"data": "..."}'